<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN_02/blob/main/DBTT_01_thiet_lap_moi_truong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sổ tay 01: Thiết lập môi trường và cấu hình dự án

**Dự án:** Nhận dạng tham số động cơ PMSM: mạng MLP nhận dạng Ld, Lq theo (id, iq); Rs tính đại số từ Ld, Lq đã nhận dạng.

**Nguyên tắc:** sổ tay chạy độc lập; mỗi lần chạy đều thiết lập lại môi trường từ đầu. Mã được viết, kiểm thử bằng dữ liệu mẫu, xuất thành tệp `.py` rồi đẩy lên GitHub theo bốn bước của PRD.

### Sơ đồ khối của sổ tay (BPMN 2.0, ISO/IEC 19510)



*Hình: Sơ đồ khối quá trình của sổ tay 01*

## Cell 1 – Thiết lập môi trường
Gắn Google Drive, nhận mã truy cập GitHub từ Colab Secrets (khóa `GITHUB_TOKEN`), sao chép kho mã về máy.

In [122]:
# Cell 1 - Thiết lập môi trường từ đầu: gắn Drive, nhận mã truy cập, sao chép kho mã
import os  # Thao tác đường dẫn và tệp
import sys  # Điều chỉnh đường dẫn nạp mô-đun
import subprocess  # Chạy lệnh git và pip
from google.colab import drive, userdata  # Gắn Drive và đọc bí mật của Colab
drive.mount('/content/drive')  # Gắn Google Drive vào phiên Colab
GITHUB_USER = 'maivu18082005-hub'  # Tài khoản GitHub chứa kho mã
REPO_NAME = 'Parameter_Identification_of_PMSM_using_PINN_02'  # Tên kho mã
REPO_DIR = f'/content/{REPO_NAME}'  # Thư mục kho trên máy Colab
TOKEN = userdata.get('GITHUB_TOKEN')  # Mã truy cập lấy từ Colab Secrets
REMOTE = f'https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'  # Địa chỉ kho kèm mã truy cập


def run(cmd, check=True):  # Hàm chạy lệnh và che mã truy cập trong đầu ra
    r = subprocess.run(cmd, capture_output=True, text=True)  # Chạy lệnh, thu đầu ra
    out = (r.stdout + r.stderr).replace(TOKEN, '***')  # Thay mã truy cập bằng dấu sao
    if check and r.returncode != 0:  # Nếu lệnh lỗi và yêu cầu kiểm tra
        raise RuntimeError(out)  # Báo lỗi kèm đầu ra đã che
    return out  # Trả về đầu ra


def push_files(paths, message):  # Hàm đẩy tệp lên GitHub
    run(['git', '-C', REPO_DIR, 'config', 'user.name', GITHUB_USER])  # Đặt tên người commit
    run(['git', '-C', REPO_DIR, 'config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com'])  # Đặt email người commit
    run(['git', '-C', REPO_DIR, 'add'] + paths)  # Thêm các tệp vào vùng chờ
    if not run(['git', '-C', REPO_DIR, 'status', '--porcelain']).strip():  # Nếu không có thay đổi
        print('Không có thay đổi để đẩy')  # Thông báo
        return  # Kết thúc hàm
    run(['git', '-C', REPO_DIR, 'commit', '-m', message])  # Tạo commit
    run(['git', '-C', REPO_DIR, 'branch', '-M', 'main'])  # Đảm bảo nhánh là main
    run(['git', '-C', REPO_DIR, 'pull', '--rebase', REMOTE, 'main'], check=False)  # Đồng bộ trước khi đẩy
    print(run(['git', '-C', REPO_DIR, 'push', REMOTE, 'HEAD:main']))  # Đẩy lên GitHub


if os.path.isdir(os.path.join(REPO_DIR, '.git')):  # Nếu kho đã có trên máy
    run(['git', '-C', REPO_DIR, 'pull', REMOTE, 'main'], check=False)  # Cập nhật kho
else:  # Nếu chưa có
    run(['git', 'clone', REMOTE, REPO_DIR])  # Sao chép kho từ GitHub
sys.path.insert(0, REPO_DIR)  # Cho phép nạp mô-đun từ kho
TESTS_OK = False  # Cờ cho phép đẩy mã, chỉ bật sau khi kiểm thử đạt
print('Kho mã sẵn sàng tại', REPO_DIR)  # Thông báo hoàn tất

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kho mã sẵn sàng tại /content/Parameter_Identification_of_PMSM_using_PINN_02


## Cell 2 – Kiểm tra và tạo cây thư mục trên Drive, kiểm tra tệp dữ liệu thô
Cây thư mục theo PRD; tệp thô phải có đủ 17 cột của bộ dữ liệu (cột `S_FW` được giữ trong tệp thô nhưng không dùng trong mô hình).

In [123]:
# Cell 2 - Tạo cây thư mục Drive và kiểm tra tệp dữ liệu thô
import pandas as pd  # Thư viện bảng dữ liệu
DRIVE_ROOT = '/content/drive/MyDrive/' + REPO_NAME  # Thư mục gốc dự án trên Drive
SUBDIRS = ['dataset/raw', 'dataset/processed', 'models', 'results', 'backups']  # Các thư mục con theo PRD
for d in SUBDIRS:  # Duyệt từng thư mục con
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)  # Tạo nếu chưa có
EXPECTED_COLS = ['Time', 'Isc_m', 'Isb_m', 'Isa_m', 'Id', 'Iq', 'S_FW', 'Idr', 'Iqr', 'nm', 'Ld', 'Lq', 'nm_ref', 'T_Load', 'ud', 'uq', 'Rs']  # Tên cột của bộ dữ liệu
RAW_FILES = ['Data1.csv', 'Data2.csv', 'Data3.csv', 'Data4.csv', 'Data5.csv']  # Danh sách tệp thô
RAW_OK = True  # Cờ báo dữ liệu thô đầy đủ
for f in RAW_FILES:  # Duyệt từng tệp thô
    p = os.path.join(DRIVE_ROOT, 'dataset/raw', f)  # Đường dẫn tệp
    if not os.path.exists(p):  # Nếu tệp chưa có
        print('THIẾU TỆP:', p)  # Thông báo thiếu
        RAW_OK = False  # Hạ cờ
        continue  # Sang tệp tiếp theo
    cols = [c.strip() for c in pd.read_csv(p, nrows=0).columns]  # Đọc tên cột
    miss = [c for c in EXPECTED_COLS if c not in cols]  # Tìm cột còn thiếu
    print(f, 'đủ cột' if not miss else f'thiếu cột {miss}')  # In kết quả kiểm tra
    RAW_OK = RAW_OK and not miss  # Cập nhật cờ
print('Dữ liệu thô đầy đủ:', RAW_OK)  # Kết luận

THIẾU TỆP: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN_02/dataset/raw/Data1.csv
THIẾU TỆP: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN_02/dataset/raw/Data2.csv
THIẾU TỆP: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN_02/dataset/raw/Data3.csv
THIẾU TỆP: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN_02/dataset/raw/Data4.csv
THIẾU TỆP: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN_02/dataset/raw/Data5.csv
Dữ liệu thô đầy đủ: False


## Cell 3 – Kiểm tra phần cứng và phiên bản thư viện
Ghi nhận cấu hình để đưa vào báo cáo.

In [124]:
# Cell 3 - Kiểm tra phần cứng và phiên bản thư viện
import platform  # Thông tin hệ thống
import torch  # Thư viện học sâu
import numpy as np  # Thư viện số học mảng
import pandas as pd  # Thư viện bảng dữ liệu
print('Python:', platform.python_version())  # Phiên bản Python
print('NumPy:', np.__version__, '| Pandas:', pd.__version__, '| Torch:', torch.__version__)  # Phiên bản thư viện
print('GPU khả dụng:', torch.cuda.is_available())  # Kiểm tra GPU
if torch.cuda.is_available():  # Nếu có GPU
    print('Tên GPU:', torch.cuda.get_device_name(0))  # In tên GPU

Python: 3.13.18
NumPy: 2.4.6 | Pandas: 2.2.3 | Torch: 2.14.1+cpu
GPU khả dụng: False


## Cell 4 – Tạo khung thư mục mã nguồn và các tệp `__init__.py`
Biến các thư mục `src/...` thành gói Python để nhập mô-đun.

In [125]:
# Cell 4 - Tạo khung thư mục mã nguồn trong kho
PKGS = ['src', 'src/data', 'src/evaluation', 'src/models', 'src/training', 'src/utils', 'src/deployment']  # Danh sách gói Python
for d in PKGS + ['configs', 'notebooks']:  # Duyệt từng thư mục cần có
    os.makedirs(os.path.join(REPO_DIR, d), exist_ok=True)  # Tạo thư mục nếu chưa có
for d in PKGS:  # Duyệt từng gói
    with open(os.path.join(REPO_DIR, d, '__init__.py'), 'w', encoding='utf-8') as f:  # Mở tệp __init__.py
        f.write('# Đánh dấu thư mục là gói Python\n')  # Ghi dòng chú thích
print('Đã tạo', len(PKGS), 'gói Python')  # Thông báo

Đã tạo 7 gói Python


## Cell 5 – Viết `configs/config.yaml`
Tệp cấu hình trung tâm: thông số động cơ, dữ liệu, mô hình, huấn luyện, bộ ước lượng Rs, chỉ tiêu đánh giá. Các giá trị đánh dấu `[TBD]` cần xác nhận.

**Cấu trúc của phương pháp:** mạng MLP nhận (id, iq) và trả về (Ld, Lq); Rs không do mạng dự đoán mà được tính đại số từ phương trình trục q với Ld, Lq của MLP, rồi lọc nhân quả. Khối `rs_estimator` có ba lớp giới hạn: loại mẫu thô ngoài dải vật lý, số mẫu hợp lệ tối thiểu, cắt đầu ra về dải vật lý; `method` (`ratio` hoặc `integral`), `filter` (`rect` hoặc `ema`), `l_smooth` (làm trơn Ld, Lq) được chọn trên Data4 ở sổ tay 06, Cell 7. Ψm = 0,23191792 Wb là hằng số cài đặt của mô phỏng, cố định, không hiệu chỉnh. Không dùng giới hạn tốc độ thay đổi vì nhãn Rs có các bước nhảy khoảng −8% và +10%.

Tên dự án và đường dẫn Drive giữ nguyên tên kho GitHub hiện có để không phá liên kết.

In [126]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/configs/config.yaml
# Tệp cấu hình trung tâm: MLP nhận dạng Ld, Lq theo (id, iq); Rs tính đại số từ Ld, Lq đã nhận dạng
project:  # Nhóm thông tin chung của dự án
  name: Parameter_Identification_of_PMSM_using_PINN  # Tên kho mã và thư mục Drive (giữ nguyên để khớp GitHub)
  seed: 42  # Hạt giống ngẫu nhiên để tái lập kết quả
paths:  # Nhóm đường dẫn trên Google Drive
  drive_root: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN_02  # Thư mục gốc trên Drive
  raw: dataset/raw  # Dữ liệu thô
  processed: dataset/processed  # Dữ liệu đã xử lý
  models: models  # Trọng số mô hình đã huấn luyện
  results: results  # Biểu đồ, nhật ký, kết quả
  backups: backups  # Bản sao lưu dự phòng
motor:  # Thông số động cơ và giới hạn vận hành
  pole_pairs: 9  # Số cặp cực p
  psi_m: 0.23191792  # Từ thông nam châm Psi_m (Wb): hằng số cài đặt của mô phỏng, cố định, không hiệu chỉnh, không làm tròn; ghi đủ 8 chữ số có nghĩa (sai số Rs xấp xỉ 200 lần sai lệch tương đối của Psi_m)
  rs_20c: 0.03383  # Điện trở pha stator tại 20 °C (Ω)
  rs_128c: 0.04828  # Điện trở pha stator tại 128 °C (Ω), theo yêu cầu cập nhật
  theta_min_c: 20.0  # Nhiệt độ cuộn dây thấp nhất (°C), ứng với rs_20c
  theta_max_c: 128.0  # Nhiệt độ cuộn dây cao nhất (°C), ứng với rs_128c
  temp_k: 232.85  # Hằng số nhiệt độ suy ra từ Rs tại 20 °C và 128 °C; IEC 60034-2-1 dùng 235 [TBD] xác nhận
  ld_ref: 0.00115  # [TBD] Giá trị tham chiếu để chuẩn hóa sai số Ld trong hàm mất mát (H), làm tròn từ trung vị Ld quan sát được; cập nhật theo tập huấn luyện
  lq_ref: 0.00143  # [TBD] Giá trị tham chiếu để chuẩn hóa sai số Lq trong hàm mất mát (H), làm tròn từ trung vị Lq quan sát được; cập nhật theo tập huấn luyện
  i_rated_peak: 65.28  # Dòng đỉnh pha định mức In,pk (A), hệ số chuẩn hóa đầu vào id, iq
  u_dc: 300.0  # Điện áp một chiều của bộ biến tần (V), dùng tính điện áp cơ sở
  n_rated: 749.45  # Tốc độ định mức (vòng/phút)
data:  # Cấu hình bộ dữ liệu
  raw_files: [Data1.csv, Data2.csv, Data3.csv, Data4.csv, Data5.csv]  # Danh sách tệp thô
  columns: [Time, Isc_m, Isb_m, Isa_m, Id, Iq, S_FW, Idr, Iqr, nm, Ld, Lq, nm_ref, T_Load, ud, uq, Rs]  # Tên 17 cột; S_FW giữ trong tệp thô, không dùng
  split:  # Chia tập theo tệp để tránh rò rỉ thông tin
    train: [Data1.csv, Data2.csv, Data3.csv]  # Tập huấn luyện
    val: [Data4.csv]  # Tập kiểm định
    test: [Data5.csv]  # Tập kiểm thử
  stride: 1  # [TBD] Bước lấy mẫu con
  sg_window: 21  # [TBD] Độ rộng cửa sổ Savitzky-Golay (số mẫu, lẻ)
  sg_polyorder: 3  # [TBD] Bậc đa thức Savitzky-Golay
  rs_step_threshold: 0.005  # Ngưỡng tương đối phát hiện bước nhảy Rs
  t_settle: 0.05  # [TBD] Thời gian xác lập sau bước nhảy Rs (s)
  bound_margin: 0.2  # Biên mở rộng khoảng giá trị đầu ra của mạng (Ld, Lq)
  consistency_rms_pu: 0.02  # [TBD] Ngưỡng RMS sai số điện áp trục q theo điện áp cơ sở; hiệu chỉnh theo dữ liệu thật
  psi_check_tol_pct: 0.01  # [TBD-11] Sai lệch tương đối tối đa (%) giữa Psi_m ước lượng từ dữ liệu và Psi_m trong cấu hình; 0,01% theo báo cáo
model:  # Cấu hình mạng MLP
  hidden_sizes: [32, 32]  # [TBD] Số nơ-ron mỗi lớp ẩn
  activation: tanh  # Hàm kích hoạt lớp ẩn
  input_names: [Id, Iq]  # Vector đầu vào
  output_names: [Ld, Lq]  # Vector đầu ra
train:  # Cấu hình huấn luyện
  batch_size: 1024  # [TBD] Kích thước lô
  epochs: 300  # [TBD] Số kỷ nguyên tối đa
  lr: 0.001  # [TBD] Tốc độ học ban đầu
  weight_decay: 0.0001  # Hệ số suy giảm trọng số
  patience: 30  # Số kỷ nguyên chờ trước khi dừng sớm
  grad_clip: 1.0  # Ngưỡng cắt gradient
rs_estimator:  # Bộ ước lượng Rs đại số có lọc: Rs tính từ phương trình trục q với Ld, Lq do MLP nhận dạng
  method: ratio  # Dạng ước lượng: ratio (Σ(num·iq)/Σ(iq²), dùng diq_dt đã làm trơn) hoặc integral (Σa/Σb, dùng hiệu lùi của iq, không cần đạo hàm làm trơn); chọn trên Data4 (sổ tay 06, Cell 7)
  filter: rect  # Kiểu cửa sổ: rect (chữ nhật, window mẫu) hoặc ema (trọng số mũ λ = 1 − 1/window, chỉ nhân quả)
  l_smooth: 0  # Số mẫu làm trơn mũ nhân quả của Ld, Lq trước khi tính Rs; 0 hoặc 1 là không làm trơn
  window: 2000  # [TBD-12] Cửa sổ lọc (số mẫu; với filter ema là hằng số thời gian); gắn với stride và chu kỳ lấy mẫu của dữ liệu; chọn trên Data4 (sổ tay 06, Cell 7)
  iq_min_a: 10.0  # [TBD-12] Ngưỡng |Iq| tối thiểu (A); mẫu dưới ngưỡng không dùng để ước lượng Rs; chọn trên Data4, không chọn trên Data5
  causal: true  # Lọc nhân quả (chỉ dùng mẫu quá khứ), phù hợp chạy thời gian thực trên DSP
  band_gate: true  # Lớp 1: loại mẫu Rs thô nằm ngoài dải vật lý trước khi lọc
  band_low_factor: 0.9  # Biên dưới của dải vật lý bằng hệ số này nhân Rs tại 20 °C
  band_high_factor: 1.1  # Biên trên của dải vật lý bằng hệ số này nhân Rs tại 128 °C
  min_valid_ratio: 0.25  # Lớp 2: chỉ cập nhật Rs khi cửa sổ có ít nhất tỷ lệ này mẫu hợp lệ (500/2.000); nếu không giữ giá trị trước, đầu tệp dùng rs_20c
  clip_output: true  # Lớp 3: cắt Rs đầu ra về dải vật lý (lưới an toàn cho DSP)
evaluation:  # Chỉ tiêu đánh giá theo PRD
  verdict_stat: max  # [TBD] Thống kê kết luận: max hoặc p95
  recon_horizon_s: 0.01  # [TBD] Độ dài đoạn mô phỏng tái tạo dòng điện (s)
  rmse_limit_pct: 2.0  # Giới hạn RMSE dòng điện (%)
  tolerances:  # Sai số tương đối tối đa cho phép (%)
    Rs: {steady: 3.0, transient: 5.0}  # Điện trở stator
    Ld: {all: 5.0}  # [TBD] Điện cảm trục d, một ngưỡng chung cho mọi mẫu (lấy ngưỡng chặt hơn của PRD)
    Lq: {all: 5.0}  # [TBD] Điện cảm trục q, một ngưỡng chung cho mọi mẫu (lấy ngưỡng chặt hơn của PRD)
deployment:  # Cấu hình triển khai trên DSP
  cpu_mhz: 200.0  # [TBD] Tần số xung nhịp DSP (MHz)
  task_period_us: 1000.0  # [TBD] Chu kỳ tác vụ nhận dạng (µs)
  budget_ratio: 0.2  # [TBD] Tỷ lệ thời gian thực thi tối đa trên chu kỳ tác vụ

Overwriting /content/Parameter_Identification_of_PMSM_using_PINN_02/configs/config.yaml


## Cell 6 – Viết `requirements.txt`
Danh sách thư viện tối thiểu.

In [127]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/requirements.txt
numpy>=1.24
pandas>=2.0
scipy>=1.10
matplotlib>=3.7
pyyaml>=6.0
torch>=2.0

Overwriting /content/Parameter_Identification_of_PMSM_using_PINN_02/requirements.txt


## Cell 7 – Viết `.gitignore`
Loại dữ liệu, trọng số và tệp tạm khỏi kho mã.

In [128]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/.gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
dataset/
*.csv
*.csv.gz
*.pt
results/
backups/

Overwriting /content/Parameter_Identification_of_PMSM_using_PINN_02/.gitignore


## Cell 8 – Viết `README.md`
Mô tả dự án, cấu trúc kho và thứ tự chạy các sổ tay.

In [129]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/README.md
# Nhận dạng tham số động cơ PMSM: MLP nhận dạng Ld, Lq và bộ ước lượng Rs đại số

Đồ án tốt nghiệp cử nhân, ngành Kỹ thuật điều khiển và Tự động hóa, Đại học Bách khoa Hà Nội.

## Mục tiêu
Nhận dạng điện cảm Ld, Lq và điện trở stator Rs của động cơ PMSM từ ud, uq, id, iq, nm.
Huấn luyện ngoại tuyến; vận hành trực tuyến chỉ thực hiện lan truyền thuận của MLP với trọng số đã đóng băng và bộ ước lượng Rs đại số.

## Phương pháp
1. Mạng MLP ánh xạ (id, iq) sang (Ld, Lq), huấn luyện có nhãn bằng sai số bình phương tương đối; đầu ra bị chặn trong khoảng nhãn của tập huấn luyện.
2. Rs không do mạng dự đoán. Rs được tính đại số từ phương trình trục q với Ld, Lq của MLP rồi lọc tỷ số các tổng (dạng `ratio` hoặc `integral`, cửa sổ `rect` hoặc `ema`),
   với ba lớp giới hạn: loại mẫu thô ngoài dải vật lý [0,9·Rs(20 °C); 1,1·Rs(128 °C)], số mẫu hợp lệ tối thiểu mỗi cửa sổ, cắt đầu ra về dải vật lý.
3. Từ thông nam châm Psi_m = 0,23191792 Wb là hằng số cài đặt của mô phỏng: cố định, không hiệu chỉnh, phải giữ đủ 8 chữ số có nghĩa.
4. Đối chứng: Rs tính bằng cùng bộ ước lượng nhưng dùng nhãn Ld, Lq, để tách sai số do MLP khỏi sai số do bộ ước lượng và dữ liệu.

## Cấu trúc kho
- `configs/config.yaml`: cấu hình trung tâm.
- `src/data/`: tiền xử lý, lớp Dataset, DataLoader.
- `src/models/`: ld_lq_mlp.py.
- `src/training/`: trainer.py (hàm mất mát và bộ huấn luyện).
- `src/evaluation/`: metrics.py (chỉ số và bộ ước lượng Rs), test.py.
- `src/deployment/`: xuất mã C cho DSP (sẽ cập nhật theo mô hình mới).
- `src/utils/`: tiện ích dùng chung.
- `notebooks/`: sáu sổ tay Colab chạy độc lập.

## Thứ tự chạy sổ tay
01 thiết lập, 02 tiền xử lý, 03 Dataset/DataLoader, 04 MLP Ld, Lq, 05 huấn luyện, 06 đánh giá (Ld, Lq, Rs).

## Chuẩn bị trên Colab
Lưu mã truy cập GitHub vào Colab Secrets với khóa `GITHUB_TOKEN` và bật quyền truy cập cho sổ tay.
Dữ liệu thô đặt tại `dataset/raw` trên Google Drive; trọng số và kết quả lưu trên Drive, không đưa lên GitHub.

Overwriting /content/Parameter_Identification_of_PMSM_using_PINN_02/README.md


## Cell 9 – Viết `src/utils/common.py`
Nạp cấu hình, đặt hạt giống, dựng đường dẫn.

In [130]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/src/utils/common.py
"""Mô-đun tiện ích dùng chung: nạp cấu hình, đặt hạt giống, dựng đường dẫn."""  # Mô tả mô-đun
import os  # Thao tác đường dẫn hệ điều hành
import random  # Bộ sinh số ngẫu nhiên chuẩn của Python
import numpy as np  # Thư viện số học mảng
import yaml  # Đọc tệp cấu hình YAML


def load_config(path):  # Hàm nạp tệp cấu hình
    """Đọc tệp YAML và trả về từ điển cấu hình."""  # Mô tả hàm
    with open(path, "r", encoding="utf-8") as f:  # Mở tệp ở chế độ đọc UTF-8
        return yaml.safe_load(f)  # Phân tích YAML an toàn rồi trả về


def set_seed(seed):  # Hàm đặt hạt giống ngẫu nhiên
    """Đặt hạt giống cho random, numpy và torch để tái lập kết quả."""  # Mô tả hàm
    random.seed(seed)  # Hạt giống cho random
    np.random.seed(seed)  # Hạt giống cho numpy
    os.environ["PYTHONHASHSEED"] = str(seed)  # Cố định băm chuỗi
    try:  # Thử nạp torch nếu có
        import torch  # Thư viện học sâu
        torch.manual_seed(seed)  # Hạt giống cho torch CPU
        torch.cuda.manual_seed_all(seed)  # Hạt giống cho mọi GPU
        torch.backends.cudnn.deterministic = True  # Buộc thuật toán xác định
        torch.backends.cudnn.benchmark = False  # Tắt tự chọn thuật toán
    except ImportError:  # Trường hợp chưa cài torch
        pass  # Bỏ qua


def get_paths(cfg):  # Hàm dựng đường dẫn tuyệt đối
    """Trả về từ điển đường dẫn tuyệt đối trên Drive từ cấu hình."""  # Mô tả hàm
    root = cfg["paths"]["drive_root"]  # Thư mục gốc dự án trên Drive
    keys = ["raw", "processed", "models", "results", "backups"]  # Các thư mục con
    return {k: os.path.join(root, cfg["paths"][k]) for k in keys}  # Ghép đường dẫn tuyệt đối

Overwriting /content/Parameter_Identification_of_PMSM_using_PINN_02/src/utils/common.py


## Cell 10 – Kiểm thử đơn vị tiện ích và cấu hình
Kiểm tra nạp cấu hình, tái lập hạt giống, đường dẫn và tính đầy đủ của tệp.

In [131]:
# Cell 10 - Kiểm thử đơn vị tiện ích và cấu hình
import importlib  # Nạp lại mô-đun vừa ghi
import numpy as np  # Thư viện số học mảng
import yaml  # Đọc tệp YAML
import os
import torch
import src.utils.common as common  # Mô-đun cần kiểm thử
importlib.reload(common)  # Nạp lại để dùng bản mới ghi

# Xác định đường dẫn tệp config.yaml hợp lệ nhất
config_path = os.path.join(REPO_DIR, 'configs', 'config.yaml')
if not os.path.exists(config_path):
    # Dự phòng nếu tệp vẫn bị ghi đè vào thư mục cũ
    fallback_path = '/content/Parameter_Identification_of_PMSM_using_PINN_02/configs/config.yaml'
    if os.path.exists(fallback_path):
        config_path = fallback_path

cfg = common.load_config(config_path)  # Nạp cấu hình
assert cfg['motor']['pole_pairs'] == 9, 'Số cặp cực phải bằng 9'  # Kiểm tra số cặp cực
assert abs(cfg['motor']['psi_m'] - 0.23191792) < 1e-12, 'Psi_m phải bằng 0,23191792 Wb'  # Kiểm tra Psi_m chính xác
assert cfg['data']['psi_check_tol_pct'] <= 0.01, 'Dung sai kiểm tra Psi_m phải không quá 0,01%'  # Kiểm tra dung sai Psi_m theo báo cáo
rs_cfg_ = cfg['rs_estimator']  # Cấu hình bộ ước lượng Rs đại số
assert int(rs_cfg_['window']) >= 1 and rs_cfg_['iq_min_a'] > 0 and isinstance(rs_cfg_['causal'], bool), 'Cấu hình rs_estimator không hợp lệ'  # Kiểm tra cấu hình bộ ước lượng
assert 0 < rs_cfg_['band_low_factor'] <= 1.0 <= rs_cfg_['band_high_factor'] and 0 < rs_cfg_['min_valid_ratio'] <= 1 and isinstance(rs_cfg_['band_gate'], bool) and isinstance(rs_cfg_['clip_output'], bool), 'Cấu hình ba lớp giới hạn Rs không hợp lệ'  # Kiểm tra các khóa
assert abs(rs_cfg_['band_low_factor'] * cfg['motor']['rs_20c'] - 0.030447) < 1e-9 and abs(rs_cfg_['band_high_factor'] * cfg['motor']['rs_128c'] - 0.053108) < 1e-9, 'Dải vật lý của Rs phải là [0,0305; 0,0531] Ω'  # Kiểm tra dải vật lý
assert rs_cfg_['method'] in ('ratio', 'integral') and rs_cfg_['filter'] in ('rect', 'ema') and int(rs_cfg_['l_smooth']) >= 0, 'method, filter, l_smooth không hợp lệ'  # Kiểm tra khóa
assert 'psi_m' not in rs_cfg_, 'Psi_m là hằng số cài đặt mô phỏng, không được đặt riêng trong rs_estimator'  # Cấm hiệu chỉnh Psi_m
assert 'rate_limit' not in rs_cfg_, 'Không dùng giới hạn tốc độ thay đổi Rs'  # Kiểm tra đã bỏ lớp 4
assert abs(cfg['motor']['rs_20c'] - 0.03383) < 1e-12, 'Rs tại 20 °C phải bằng 0,03383 Ω'  # Kiểm tra Rs tại 20 °C
assert abs(cfg['motor']['rs_128c'] - 0.04828) < 1e-12, 'Rs tại 128 °C phải bằng 0,04828 Ω'  # Kiểm tra Rs tại 128 °C
assert cfg['motor']['theta_min_c'] == 20.0 and cfg['motor']['theta_max_c'] == 128.0, 'Dải nhiệt độ phải là 20 đến 128 °C'  # Kiểm tra dải nhiệt độ
assert set(cfg['evaluation']['tolerances']['Ld']) == {'all'} and set(cfg['evaluation']['tolerances']['Lq']) == {'all'}, 'Dung sai Ld, Lq phải là một ngưỡng chung'  # Kiểm tra dung sai
assert cfg['motor']['ld_ref'] > 0 and cfg['motor']['lq_ref'] > 0, 'Giá trị tham chiếu Ld, Lq phải dương'  # Kiểm tra tham chiếu
assert cfg['data']['columns'] == EXPECTED_COLS and len(EXPECTED_COLS) == 17, 'Tên cột phải khớp bộ dữ liệu 17 cột'  # Kiểm tra tên cột
assert cfg['data']['split'] == {'train': ['Data1.csv', 'Data2.csv', 'Data3.csv'], 'val': ['Data4.csv'], 'test': ['Data5.csv']}, 'Chia tập phải là 1-2-3 / 4 / 5'  # Kiểm tra chia tập
assert cfg['model']['input_names'] == ['Id', 'Iq'] and cfg['model']['output_names'] == ['Ld', 'Lq'], 'MLP phải nhận (Id, Iq) và trả về (Ld, Lq)'  # Kiểm tra đầu vào/ra
assert 'loss' not in cfg and 'n_max' not in cfg['motor'], 'Còn khóa cấu hình của mô hình cũ'  # Kiểm tra đã bỏ khóa cũ
common.set_seed(7)  # Đặt hạt giống lần một
a1 = np.random.rand(5)  # Sinh số ngẫu nhiên lần một
t1 = torch.rand(5)  # Sinh tensor ngẫu nhiên lần một
common.set_seed(7)  # Đặt lại cùng hạt giống
assert np.allclose(a1, np.random.rand(5)), 'NumPy chưa tái lập'  # Kiểm tra tái lập NumPy
assert torch.equal(t1, torch.rand(5)), 'Torch chưa tái lập'  # Kiểm tra tái lập Torch
P = common.get_paths(cfg)  # Dựng đường dẫn
assert set(P) == {'raw', 'processed', 'models', 'results', 'backups'}, 'Thiếu khóa đường dẫn'  # Kiểm tra khóa
assert all(os.path.isdir(v) for v in P.values()), 'Thư mục Drive chưa tồn tại'  # Kiểm tra thư mục tồn tại
for rel in ['requirements.txt', '.gitignore', 'README.md'] + [d + '/__init__.py' for d in PKGS]:  # Duyệt các tệp bắt buộc
    path_to_check = os.path.join(REPO_DIR, rel)
    if not os.path.exists(path_to_check):
         path_to_check = os.path.join('/content/Parameter_Identification_of_PMSM_using_PINN_02', rel)
    assert os.path.exists(path_to_check), f'Thiếu tệp {rel}'  # Kiểm tra tồn tại
TESTS_OK = True  # Bật cờ cho phép đẩy mã
print('Toàn bộ kiểm thử đơn vị đạt')

Toàn bộ kiểm thử đơn vị đạt


## Cell 11 – Dọn tệp của phiên bản cũ
Gỡ khỏi kho các mô-đun không còn dùng (mạng cơ sở nhiều đầu vào, mô hình PINN, hàm mất mát vật lý); lần đẩy ở Cell 12 sẽ ghi nhận việc xóa.

In [132]:
# Cell 11 - Gỡ các tệp của phiên bản cũ khỏi kho
OBSOLETE = ['src/models/baselineMLP.py', 'src/models/pinn_model.py', 'src/training/loss_functions.py']  # Các mô-đun không còn dùng
removed = []  # Danh sách tệp đã gỡ
for rel in OBSOLETE:  # Duyệt từng tệp
    p = os.path.join(REPO_DIR, rel)  # Đường dẫn trong kho
    if os.path.exists(p):  # Nếu tệp còn tồn tại
        run(['git', '-C', REPO_DIR, 'rm', '-q', '-f', '--ignore-unmatch', rel])  # Gỡ khỏi chỉ mục git
        if os.path.exists(p):  # Nếu tệp chưa được theo dõi thì xóa trực tiếp
            os.remove(p)  # Xóa tệp
        removed.append(rel)  # Ghi nhận
print('Đã gỡ:', removed if removed else 'không có tệp cũ')  # Thông báo

Đã gỡ: không có tệp cũ


## Cell 12 – Bước 4: đẩy lên GitHub

In [133]:
# Cell cuối - Bước 4: đẩy tệp đã kiểm thử đạt lên GitHub
assert TESTS_OK, 'Chưa đạt toàn bộ kiểm thử, không được đẩy mã'  # Chặn đẩy mã khi kiểm thử chưa đạt
push_files(['configs/config.yaml', 'requirements.txt', '.gitignore', 'README.md', 'src'], 'Chuyển sang MLP nhận dạng Ld, Lq và Rs đại số: cấu hình, tiện ích, README')  # Đẩy các tệp lên nhánh main

Không có thay đổi để đẩy
